# Atividade 1 - Pipeline RAG sobre PDF

Este notebook implementa um sistema completo de **Perguntas e Respostas** baseado em um documento PDF armazenado no Google Drive, utilizando a abordagem de **RAG (*Retrieval-Augmented Generation*)**.

O objetivo é demonstrar, de forma didática e modular, como construir um pipeline de PLN capaz de **responder perguntas com base exclusiva no conteúdo do documento**, preservando rastreabilidade e controle das fontes utilizadas.

O pipeline implementado segue as seguintes etapas:

1. **Ingestão do PDF**  
   Leitura do documento e extração de texto por página, com preservação de metadados (ex.: número da página e origem).

2. **Normalização mínima do texto**  
   Remoção de ruídos comuns da extração de PDFs (hifenização invisível, espaços excessivos, quebras artificiais), sem alterar o conteúdo semântico.

3. **Fragmentação (*chunking*) por tokens**  
   Divisão do texto em fragmentos semanticamente coerentes, utilizando contagem de tokens (via *tiktoken*), adequada ao uso posterior por modelos de linguagem.

4. **Geração de embeddings e indexação vetorial**  
   Conversão dos fragmentos em vetores semânticos por meio da API de *embeddings* da OpenAI e indexação eficiente com **FAISS**, incluindo persistência em cache no Google Drive.

5. **Recuperação de evidências e resposta gerativa**  
   Recuperação dos fragmentos mais relevantes para uma pergunta e geração da resposta por um modelo de linguagem via **OpenAI Responses API**, com **citações explícitas das fontes utilizadas**.



## 0. Configuração da chave na OpenAI


### 1. Criando a chave na plataforma da OpenAI

Para utilizar os serviços da API da OpenAI (embeddings e geração de respostas), é necessário criar uma chave de acesso pessoal (`OPENAI_API_KEY`).

1. Acesse: **https://platform.openai.com/**
2. Faça login ou crie uma conta.
3. No menu lateral, selecione **API Keys**.
4. Clique em **Create new secret key**.
5. Em **Owned by**, escolha **You** (chave pessoal, adequada para uso didático).
6. Em **Permissions**, selecione **All**.
7. Clique em **Create secret key**.
8. Copie a chave gerada.  

<br>

### 2. Definindo a chave no Google Colab (Secrets)

A chave **não deve ser escrita diretamente no código**, nem compartilhada em notebooks públicos.

No Google Colab, utilizamos o gerenciador de segredos:

1. No painel lateral esquerdo, clique no ícone 🔑 **Secrets**.
2. Clique em **Adicionar novo secret** e preencha:
   - **Name:** `OPENAI_API_KEY`  
   - **Value:** *(cole aqui a sua chave)*  
   - **Habilite:** *Acesso ao notebook*
3. Salve o secret.

Após esse procedimento, a variável de ambiente `OPENAI_API_KEY` estará automaticamente disponível para o notebook, permitindo o uso da API da OpenAI com segurança.

<br>

### 3. Verificando da variável de ambiente `OPENAI_API_KEY`

Antes de executar o pipeline, verificamos se a variável de ambiente
`OPENAI_API_KEY` está definida no ambiente do Google Colab.

Essa verificação evita erros silenciosos de autenticação e garante que
a API da OpenAI possa ser utilizada corretamente.


In [13]:
import os
from google.colab import userdata
from google.colab.userdata import SecretNotFoundError, NotebookAccessError

def load_openai_key_from_colab(key_name:str):
    try:
        key = userdata.get(key_name)
        os.environ[key_name] = key
        print(f"✅ {key_name} carregada com sucesso a partir do Colab Secrets.")
    except (SecretNotFoundError, NotebookAccessError):
        print(f"⚠️ {key_name} não configurada corretamente no Colab Secrets.")

## 1. Instalação e imports

Nesta etapa, instalamos as dependências e importamos os pacotes necessários para:
- carregar o PDF (LangChain + PyPDF);
- fragmentar texto (LangChain Text Splitters);
- tokenizar (tiktoken);
- indexar e buscar vetores (FAISS);
- gerar embeddings e respostas (OpenAI API).


In [14]:
!pip -q install -U pypdf langchain-community langchain-text-splitters tiktoken faiss-cpu openai

from __future__ import annotations

import os
import re
import json
import pickle
import hashlib
from dataclasses import dataclass, field
from datetime import datetime
from typing import Dict, Any, List, Optional, Tuple

import numpy as np
import faiss
import tiktoken

from openai import OpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document

## 2. Infraestrutura de acesso ao Google Drive

Esta seção define funções utilitárias reutilizáveis para:

- montar o Google Drive no ambiente Colab;
- resolver caminhos relativos ao `MyDrive`;
- validar a existência de arquivos.

Nenhuma execução específica do projeto ocorre aqui.
A seleção do PDF e a construção do sistema são feitas apenas na Seção 11.



In [15]:
import os
from google.colab import drive

DRIVE_ROOT = "/content/drive/MyDrive"

def mount_drive(force: bool = False) -> None:
    """
    Monta o Google Drive no ambiente Colab.
    """
    drive.mount("/content/drive", force_remount=force)

def resolve_drive_path(relative_path: str, base: str = DRIVE_ROOT) -> str:
    """
    Constrói o caminho absoluto a partir de um caminho relativo ao MyDrive
    e valida a existência do arquivo.

    Exemplo:
        relative_path = "fatec/.../arquivo.pdf"
    """
    full_path = os.path.join(base, relative_path)
    if not os.path.isfile(full_path):
        raise FileNotFoundError(f"Arquivo não encontrado: {full_path}")
    return full_path

## 3. Configuração e estado do sistema

Para modularizar o projeto, definimos duas estruturas:

- `RAGConfig`: concentra parâmetros (modelos, chunking, recuperação e cache).
- `RAGState`: armazena os artefatos construídos pelo pipeline (documentos, chunks, índice FAISS etc.).

Isso permite encadear funções com clareza e manter rastreabilidade para auditoria e correção.


In [16]:
@dataclass
class RAGConfig:
    # OpenAI
    embed_model: str = "text-embedding-3-small"
    gen_model: str = "gpt-4.1-mini"

    # Chunking (tokens)
    chunk_size_tokens: int = 220
    chunk_overlap_tokens: int = 40
    tokenizer_name: str = "cl100k_base"

    # Retrieval
    top_k: int = 8
    min_score: float = 0.25
    max_context_chars: int = 8000

    # Cache
    force_rebuild: bool = False
    cache_dir_mode: str = "same_folder"  # "same_folder" ou "fixed"
    fixed_cache_dir: str = "/content/drive/MyDrive/pln_rag_cache"

    # Resposta
    temperature: float = 0.0


@dataclass
class RAGState:
    pdf_path: str
    config: RAGConfig
    client: OpenAI

    documents: List[Document] = field(default_factory=list)
    chunks: List[Document] = field(default_factory=list)
    chunk_store: Dict[int, Dict[str, Any]] = field(default_factory=dict)
    chunk_ids: List[int] = field(default_factory=list)

    index: Optional[faiss.Index] = None

    index_dir: Optional[str] = None
    fingerprint: Optional[Dict[str, Any]] = None

## 4. Ingestão e normalização do PDF

Esta etapa converte o PDF em uma lista de `Document` (um por página), contendo:
- `page_content`: texto extraído;
- `metadata`: metadados (ex.: página, fonte).

Em seguida, aplicamos uma normalização mínima para reduzir ruídos de extração (sem alterar o conteúdo semântico).


In [17]:
def normalize_text(text: str) -> str:
    """Normalização textual mínima e segura."""
    text = text.replace("\u00ad", "")               # soft hyphen
    text = re.sub(r"[ \t]+", " ", text)             # múltiplos espaços
    text = re.sub(r"\n{3,}", "\n\n", text)          # excesso de quebras
    return text.strip()

def ingest_pdf(pdf_path: str) -> List[Document]:
    """Carrega PDF como Documents (um por página) e normaliza o texto."""
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()
    if not docs:
        raise ValueError("Nenhuma página carregada do PDF.")
    for d in docs:
        d.page_content = normalize_text(d.page_content)
        d.metadata.setdefault("source", pdf_path)
    return docs


## 5. Chunking robusto por tokens

Fragmentar por tokens tende a ser mais robusto para sistemas RAG do que por caracteres, pois aproxima melhor:
- limites de contexto utilizados por modelos;
- preservação de unidades linguísticas.

Aqui usamos `RecursiveCharacterTextSplitter` com uma função de comprimento baseada em `tiktoken`.
Também inserimos `chunk_id` nos metadados para rastreabilidade.


In [18]:
def _tiktoken_len_fn(encoding):
    def _len(text: str) -> int:
        return len(encoding.encode(text))
    return _len

def split_documents_token_based(
    docs: List[Document],
    chunk_size_tokens: int,
    chunk_overlap_tokens: int,
    tokenizer_name: str,
) -> List[Document]:
    from langchain_text_splitters import RecursiveCharacterTextSplitter

    enc = tiktoken.get_encoding(tokenizer_name)
    length_fn = _tiktoken_len_fn(enc)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size_tokens,
        chunk_overlap=chunk_overlap_tokens,
        length_function=length_fn,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    chunks = splitter.split_documents(docs)

    for i, ch in enumerate(chunks):
        ch.metadata["chunk_id"] = i
        ch.metadata.setdefault("source", ch.metadata.get("source"))
        ch.metadata.setdefault("page", ch.metadata.get("page", None))

    return chunks

## 6. Repositório canônico dos chunks (`chunk_store`)

O FAISS armazena apenas vetores. Para responder e citar evidências, precisamos de um repositório canônico do conteúdo.

Criamos:
- `chunk_store`: `chunk_id → {"text", "metadata"}`
- `chunk_ids`: lista ordenada de ids (mapeia índices locais do FAISS para `chunk_id`)
- `texts`: lista de textos alinhada a `chunk_ids` (ordem usada na vetorização).


In [19]:
def build_chunk_store(chunks: List[Document]) -> Tuple[Dict[int, Dict[str, Any]], List[int], List[str]]:
    store: Dict[int, Dict[str, Any]] = {}
    for ch in chunks:
        cid = ch.metadata.get("chunk_id")
        if cid is None:
            raise ValueError("chunk_id ausente no chunk. Verifique a etapa de chunking.")
        store[int(cid)] = {"text": ch.page_content, "metadata": dict(ch.metadata)}

    chunk_ids = sorted(store.keys())
    texts = [store[cid]["text"] for cid in chunk_ids]
    return store, chunk_ids, texts

## 7. Embeddings e construção do índice FAISS

Nesta etapa:
1. geramos embeddings para cada chunk via OpenAI (em lotes);
2. normalizamos os vetores (L2) para trabalhar com similaridade por cosseno;
3. criamos um índice FAISS `IndexFlatIP` (produto interno). Com vetores normalizados, ele equivale ao cosseno.


In [20]:
def embed_texts_openai(client: OpenAI, texts: List[str], model: str, batch_size: int = 128) -> np.ndarray:
    vectors: List[List[float]] = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        resp = client.embeddings.create(model=model, input=batch)
        vectors.extend([item.embedding for item in resp.data])
    return np.array(vectors, dtype=np.float32)

def build_faiss_index(embeddings: np.ndarray) -> faiss.Index:
    faiss.normalize_L2(embeddings)
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    return index

## 8. Persistência do índice no Drive (FAISS + store + fingerprint)

Para evitar recomputar embeddings a cada execução, salvamos no Drive:
- `faiss.index`
- `chunk_store.pkl`
- `chunk_ids.pkl`
- `config.json` (com fingerprint e metadados)

O cache é validado por um `fingerprint` que inclui:
- hash SHA-256 do PDF;
- modelo de embeddings;
- parâmetros de chunking e tokenizer.

Se o fingerprint mudar, o índice é reconstruído automaticamente.


In [21]:
def sha256_file(path: str, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def get_index_dir(pdf_path: str, config: RAGConfig) -> str:
    if config.cache_dir_mode == "fixed":
        os.makedirs(config.fixed_cache_dir, exist_ok=True)
        pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
        d = os.path.join(config.fixed_cache_dir, f"{pdf_name}_rag_index")
        os.makedirs(d, exist_ok=True)
        return d

    pdf_dir = os.path.dirname(pdf_path)
    pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
    d = os.path.join(pdf_dir, f"{pdf_name}_rag_index")
    os.makedirs(d, exist_ok=True)
    return d

def make_fingerprint(pdf_path: str, config: RAGConfig) -> Dict[str, Any]:
    return {
        "pdf_path": pdf_path,
        "pdf_sha256": sha256_file(pdf_path),
        "embed_model": config.embed_model,
        "chunk_size_tokens": config.chunk_size_tokens,
        "chunk_overlap_tokens": config.chunk_overlap_tokens,
        "tokenizer_name": config.tokenizer_name,
    }

def cache_paths(index_dir: str) -> Dict[str, str]:
    return {
        "faiss": os.path.join(index_dir, "faiss.index"),
        "store": os.path.join(index_dir, "chunk_store.pkl"),
        "ids": os.path.join(index_dir, "chunk_ids.pkl"),
        "cfg": os.path.join(index_dir, "config.json"),
    }

def cache_exists(index_dir: str) -> bool:
    p = cache_paths(index_dir)
    return all(os.path.isfile(p[k]) for k in p)

def save_cache(index_dir: str, index: faiss.Index, chunk_store: dict, chunk_ids: list, fingerprint: dict) -> None:
    p = cache_paths(index_dir)
    faiss.write_index(index, p["faiss"])
    with open(p["store"], "wb") as f:
        pickle.dump(chunk_store, f)
    with open(p["ids"], "wb") as f:
        pickle.dump(chunk_ids, f)

    cfg = {
        "fingerprint": fingerprint,
        "saved_at": datetime.now().isoformat(timespec="seconds"),
        "faiss_ntotal": int(index.ntotal),
    }
    with open(p["cfg"], "w", encoding="utf-8") as f:
        json.dump(cfg, f, ensure_ascii=False, indent=2)

def load_cache(index_dir: str) -> Tuple[faiss.Index, dict, list, dict]:
    p = cache_paths(index_dir)
    index = faiss.read_index(p["faiss"])
    with open(p["store"], "rb") as f:
        store = pickle.load(f)
    with open(p["ids"], "rb") as f:
        ids = pickle.load(f)
    with open(p["cfg"], "r", encoding="utf-8") as f:
        cfg = json.load(f)
    return index, store, ids, cfg

## 9. Recuperação (consulta → embedding → FAISS → trechos)

A recuperação recebe uma pergunta e retorna os chunks mais relevantes:
- gera embedding da pergunta com o mesmo modelo;
- normaliza o vetor (L2);
- consulta o FAISS (Top-k);
- aplica `min_score` para evitar respostas sem evidência.

Retorna trechos contendo score, chunk_id, texto e metadados (incluindo página e fonte).


In [22]:
def embed_query(client: OpenAI, query: str, model: str) -> np.ndarray:
    resp = client.embeddings.create(model=model, input=[query])
    emb = resp.data[0].embedding
    q_vec = np.array([emb], dtype=np.float32)
    faiss.normalize_L2(q_vec)
    return q_vec

def search_chunks(state: RAGState, query: str, top_k: Optional[int] = None, min_score: Optional[float] = None) -> List[Dict[str, Any]]:
    if state.index is None:
        raise ValueError("Índice FAISS não inicializado. Execute build_rag(...).")

    top_k = top_k if top_k is not None else state.config.top_k
    min_score = min_score if min_score is not None else state.config.min_score

    q_vec = embed_query(state.client, query, state.config.embed_model)
    scores, idxs = state.index.search(q_vec, top_k)

    results: List[Dict[str, Any]] = []
    for score, local_idx in zip(scores[0], idxs[0]):
        if local_idx == -1:
            continue
        score = float(score)
        if score < min_score:
            continue

        cid = state.chunk_ids[int(local_idx)]
        item = state.chunk_store[cid]
        meta = item.get("metadata", {})
        page = meta.get("page", None)
        page_1idx = page + 1 if isinstance(page, int) else None

        results.append({
            "score": score,
            "chunk_id": cid,
            "text": item.get("text", ""),
            "metadata": meta,
            "page": page_1idx,
            "source": os.path.basename(str(meta.get("source", "PDF"))),
        })

    return results

## 10. Orquestração e interface do sistema

Esta seção fecha o pipeline em funções encadeáveis:

- `build_rag(pdf_path, ...)`: executa ingestão → chunking → store → (cache ou rebuild) e devolve um `RAGState` pronto.
- `answer(state, question)`: faz recuperação + monta contexto com fontes + chama o modelo gerativo com regras e citações.
- `interactive_chat(state)`: loop interativo no Colab com comando `sair`.

## 11. Execução do sistema RAG

Nesta seção realizamos a execução concreta do projeto:

1. Montamos o Google Drive.
2. Definimos o caminho do PDF (relativo ao `MyDrive`).
3. Resolvemos o caminho absoluto (`pdf_path`).
4. Construímos o sistema RAG (com cache, se disponível).
5. Iniciamos o modo interativo de perguntas e respostas.

Esta é a **única seção que precisa ser alterada** quando o PDF muda.

In [28]:
def format_context(passages: List[Dict[str, Any]], max_chars: int) -> str:
    blocks = []
    total = 0
    for i, p in enumerate(passages, start=1):
        page = p.get("page")
        page_str = f"p.{page}" if isinstance(page, int) else "p.?"
        source = p.get("source", "PDF")
        block = f"[Fonte {i} | {page_str} | {source}]\n{p['text']}"
        if total + len(block) > max_chars:
            break
        blocks.append(block)
        total += len(block)
    return "\n\n".join(blocks)

def answer(state: RAGState, question: str) -> str:
    passages = search_chunks(state, question)
    if not passages:
        return "Não encontrei evidências suficientes no PDF para responder a essa pergunta com segurança."

    context = format_context(passages, max_chars=state.config.max_context_chars)

    instructions = (
        "Você é um assistente de perguntas e respostas.\n"
        "Responda APENAS com base nas fontes fornecidas.\n"
        "Se a resposta não estiver nas fontes, diga explicitamente que não é possível responder com segurança.\n"
        "Responda em português, de forma objetiva.\n"
        "Sempre cite as fontes usadas no formato [Fonte X]."
    )

    user_input = f"Pergunta:\n{question}\n\nFontes:\n{context}"

    resp = state.client.responses.create(
        model=state.config.gen_model,
        instructions=instructions,
        input=user_input,
        temperature=state.config.temperature,
    )
    return resp.output_text

def build_rag(pdf_path: str, config: Optional[RAGConfig] = None, force_rebuild: Optional[bool] = None) -> RAGState:
    config = config or RAGConfig()
    if force_rebuild is not None:
        config.force_rebuild = force_rebuild

    client = OpenAI()
    state = RAGState(pdf_path=pdf_path, config=config, client=client)

    # 1) ingestão
    state.documents = ingest_pdf(pdf_path)

    # 2) chunking
    state.chunks = split_documents_token_based(
        state.documents,
        chunk_size_tokens=config.chunk_size_tokens,
        chunk_overlap_tokens=config.chunk_overlap_tokens,
        tokenizer_name=config.tokenizer_name,
    )

    # 3) store
    state.chunk_store, state.chunk_ids, texts = build_chunk_store(state.chunks)

    # 4) cache
    state.index_dir = get_index_dir(pdf_path, config)
    state.fingerprint = make_fingerprint(pdf_path, config)

    if (not config.force_rebuild) and cache_exists(state.index_dir):
        try:
            idx, store, ids, cfg = load_cache(state.index_dir)
            fp_old = cfg.get("fingerprint", {})
            if fp_old == state.fingerprint:
                state.index = idx
                state.chunk_store = store
                state.chunk_ids = ids
                return state
        except Exception:
            pass  # fallback: rebuild

    # 5) rebuild
    embs = embed_texts_openai(client, texts, model=config.embed_model, batch_size=128)
    state.index = build_faiss_index(embs)
    save_cache(state.index_dir, state.index, state.chunk_store, state.chunk_ids, state.fingerprint)
    return state

def interactive_chat(state: RAGState, show_sources: bool = True) -> None:
    EXIT = {"sair", "exit", "quit", "q"}
    print("Sistema pronto. Digite perguntas. Para sair, digite 'sair'.")
    print("-" * 60)

    while True:
        print("\n(Aguardando pergunta... digite e pressione Enter)")
        q = input("Você: ").strip()
        if not q:
            continue
        if q.lower() in EXIT:
            print("Encerrando.")
            break
        passages = search_chunks(state, q)
        if show_sources:
            print("\nFontes recuperadas:")
            for i, p in enumerate(passages, start=1):
                page = p.get("page")
                page_str = f"p.{page}" if isinstance(page, int) else "p.?"
                print(f"- [Fonte {i}] {p.get('source')} | {page_str} | chunk_id={p.get('chunk_id')} | score={p.get('score'):.3f}")

        print("\nAssistente:", answer(state, q))
        print("-" * 60)

In [29]:
# Ajuste/garanta que pdf_path já existe e aponta para o PDF no Drive.
# Exemplo:
# pdf_path = "/content/drive/MyDrive/fatec/.../disciplinas.pdf"

# 0) Carregar a chave de acesso a sua conta na plataforma da OpenAI
load_openai_key_from_colab("OPENAI_API_KEY")

# 1) Montar Drive
mount_drive(force=False)

# 2) Definir o PDF (relativo ao MyDrive)
relative_pdf_path = "fatec/jacarei/DSM/PLN/atividades/Atividade 1/disciplinas-modalidade.pdf"

# 3) Resolver e validar
pdf_path = resolve_drive_path(relative_pdf_path)

print(f"PDF selecionado: {pdf_path}")

# 4) Construir o sistema RAG (usa cache se existir)
rag = build_rag(pdf_path, force_rebuild=False)

# 5) Chat interativo
interactive_chat(rag, show_sources=True)

✅ OPENAI_API_KEY carregada com sucesso a partir do Colab Secrets.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PDF selecionado: /content/drive/MyDrive/fatec/jacarei/DSM/PLN/atividades/Atividade 1/disciplinas-modalidade.pdf
Sistema pronto. Digite perguntas. Para sair, digite 'sair'.
------------------------------------------------------------

(Aguardando pergunta... digite e pressione Enter)
Você: qual semestre eu estudarei Iot?

Fontes recuperadas:
- [Fonte 1] disciplinas-modalidade.pdf | p.3 | chunk_id=7 | score=0.568
- [Fonte 2] disciplinas-modalidade.pdf | p.3 | chunk_id=6 | score=0.546
- [Fonte 3] disciplinas-modalidade.pdf | p.1 | chunk_id=1 | score=0.544
- [Fonte 4] disciplinas-modalidade.pdf | p.2 | chunk_id=3 | score=0.533
- [Fonte 5] disciplinas-modalidade.pdf | p.3 | chunk_id=8 | score=0.530
- [Fonte 6] disciplinas-modalidade.pdf | p.2 | chunk_id=2 | score=0.529
- [Fonte 7] disciplinas-modali